# Missingness-Aware TabNet (MA-TabNet)

* **Objective:** Implement an advanced TabNet architecture utilizing selective missingness masking.
* **Key Libraries:**
  * `pytorch-tabnet`: Core implementation for TabNet.
  * `optuna`: Automated hyperparameter optimization.

In [ ]:
# ==========================================
# [1] Required package installation
# ==========================================
!pip install pytorch-tabnet pytorch-widedeep optuna shap seaborn -q

import pandas as pd
import numpy as np
import torch
import io
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TabNet Model
from pytorch_tabnet.tab_model import TabNetRegressor

## 1. Data Loading and Preprocessing (MA-TabNet)

* **Selective Masking Strategy:**
  * Generates missing-value masks *only* for variables with missingness exceeding a predefined threshold (e.g., 5%).
  * Preserves informative missingness patterns without adding unnecessary indicator noise.
* **Imputation:**
  * Missing numerical values are imputed using the median.
  * Missing categorical values are assigned an "Unknown" class.

In [ ]:
# ==========================================
# [1] Data loading and preprocessing
# ==========================================
def load_and_prep_ma_tabnet_advanced():
    # 1. Load dataset
    try:
        from google.colab import files
        print("=== [Google Colab] Please upload Dataset.csv ===")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    except ImportError:
        print("=== [Local Environment] Loading Dataset.csv ===")
        df = pd.read_csv('Dataset.csv')

    # 2. Basic preprocessing
    if 'Country Name' in df.columns:
        df = df.drop('Country Name', axis=1)

    target = 'Maternal Mortality Ratio'
    MISSING_THRESHOLD = 0.05  # 5% threshold for generating missingness masks

    cat_cols = ['Country Code', 'Continent']
    exclude = cat_cols + ['Year', target]
    num_cols = [c for c in df.columns if c not in exclude]

    added_masks = []

    # Apply selective masking
    for col in num_cols:
        missing_ratio = df[col].isnull().mean()
        if missing_ratio >= MISSING_THRESHOLD:
            mask_col_name = f"{col}_is_missing"
            # Treat mask as a numerical feature (0.0 / 1.0) for efficient TabNet processing
            df[mask_col_name] = df[col].isnull().astype(float)
            added_masks.append(mask_col_name)

    # Impute missing values
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in cat_cols:
        df[col] = df[col].fillna("Unknown")

    # 3. Categorical encoding
    categorical_dims = {}
    for col in cat_cols:
        l_enc = LabelEncoder()
        df[col] = l_enc.fit_transform(df[col].astype(str).values)
        categorical_dims[col] = len(l_enc.classes_)

    # Reconstruct feature list
    features = [c for c in df.columns if c not in ['Year', target]]
    cat_idxs = [i for i, f in enumerate(features) if f in cat_cols]
    cat_dims = [categorical_dims[f] for f in features if f in cat_cols]

    print(f"\n[Model Info] Total input features: {len(features)}")
    print(f"[Model Info] Selectively added masks: {len(added_masks)}")

    # Temporal data splits
    def to_xy(data):
        return data[features].values, data[target].values.reshape(-1, 1)

    train_p1 = df[(df['Year'] >= 2011) & (df['Year'] <= 2014)]
    val_p1   = df[df['Year'] == 2015]
    train_p2 = df[(df['Year'] >= 2011) & (df['Year'] <= 2015)]
    test     = df[df['Year'] == 2016]

    return (to_xy(train_p1), to_xy(val_p1), to_xy(train_p2), to_xy(test),
            cat_idxs, cat_dims, features, added_masks)

# Prepare data arrays
(data_train_1, data_val_1, data_train_2, data_test,
 cat_idxs, cat_dims, feature_names, added_masks) = load_and_prep_ma_tabnet_advanced()

X_train_1, y_train_1 = data_train_1
X_val_1, y_val_1     = data_val_1
X_train_2, y_train_2 = data_train_2
X_test, y_test       = data_test

## 2. Hyperparameter Optimization

* **Optuna Integration:** * Searches for the optimal architecture parameters (`n_d`, `n_a`, `n_steps`, etc.).
  * Capacity is slightly expanded to accommodate the selective missingness masks.

In [ ]:
# ==========================================
# [2] Optuna-based hyperparameter search
# ==========================================
def objective(trial):
    param = {
        'n_d': trial.suggest_int('n_d', 16, 64, step=8),
        'n_a': trial.suggest_int('n_a', 16, 64, step=8),
        'n_steps': trial.suggest_int('n_steps', 3, 8),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'n_independent': trial.suggest_int('n_independent', 2, 5),
        'n_shared': trial.suggest_int('n_shared', 2, 5),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-4, 1e-2, log=True),
        'optimizer_params': {'lr': trial.suggest_float('lr', 1e-2, 5e-2)},
    }

    clf = TabNetRegressor(
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=1,
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        verbose=0,
        **param
    )

    clf.fit(
        X_train=X_train_1, y_train=y_train_1,
        eval_set=[(X_val_1, y_val_1)],
        eval_name=['valid'],
        eval_metric=['mae'],
        max_epochs=150,
        patience=20,
        batch_size=256,
        virtual_batch_size=128,
        drop_last=False
    )
    return clf.best_cost

print("\n>>> [Step 1] Starting MA-TabNet optimization...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

best_params = study.best_trial.params
print("\n>>> [Result] Best hyperparameters:", best_params)

## 3. Final Model Training & Bootstrap Function

* **Retraining:** * The model is retrained on the full pretest period (2011-2015) using the best parameters found by Optuna.
* **Bootstrap Confidence Intervals:**
  * Defines a robust statistical function to compute 95% Confidence Intervals for MAE, RMSE, and R2 using 1,000 resampling iterations.

In [ ]:
# ==========================================
# [3] Final training & Bootstrap function definition
# ==========================================
print("\n>>> [Step 2] Training final MA-TabNet model...")

final_params = best_params.copy()
lr = final_params.pop('lr')

clf_final = TabNetRegressor(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,
    optimizer_params={'lr': lr},
    scheduler_params={"step_size": 10, "gamma": 0.95},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    verbose=0, # Reduced verbosity for clean output
    **final_params
)

clf_final.fit(
    X_train=X_train_2, y_train=y_train_2,
    eval_set=None,
    max_epochs=200,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

def calculate_bootstrap_ci(y_true, y_pred, n_bootstraps=1000, ci=95, random_state=42):
    """
    Calculate mean performance and confidence intervals using bootstrapping.

    Parameters:
    - y_true: Ground truth target values (1D array)
    - y_pred: Model predictions (1D array)
    - n_bootstraps: Resampling iterations (default 1000)
    - ci: Confidence interval percentage (default 95%)

    Returns:
    - Dictionary with strings 'Mean (Lower_Bound - Upper_Bound)'
    """
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    n_samples = len(y_true)
    rng = np.random.RandomState(random_state)

    boot_mae, boot_rmse, boot_r2 = [], [], []

    for _ in range(n_bootstraps):
        indices = rng.randint(0, n_samples, n_samples)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        boot_mae.append(mean_absolute_error(y_true_boot, y_pred_boot))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        boot_r2.append(r2_score(y_true_boot, y_pred_boot))

    lower_p = (100 - ci) / 2.0
    upper_p = 100 - lower_p

    results = {}
    metrics = {'MAE': boot_mae, 'RMSE': boot_rmse, 'R2': boot_r2}

    for name, values in metrics.items():
        mean_val = np.mean(values)
        lower_val = np.percentile(values, lower_p)
        upper_val = np.percentile(values, upper_p)
        results[name] = f"{mean_val:.4f} ({lower_val:.4f} - {upper_val:.4f})"

    return results

## 4. Evaluation & Feature Importance

* **Held-out Test Performance:**
  * Evaluates predictions on the unseen 2016 dataset.
  * Outputs Bootstrap 95% Confidence Intervals.
* **Feature Importance Plot:**
  * Visualizes the top 15 drivers of the model.
  * Highlights selectively added missingness-mask variables in red.

In [ ]:
# ==========================================
# [4] Final evaluation and Bootstrap CI output
# ==========================================
preds = clf_final.predict(X_test)

print("\n" + "="*50)
print(" MA-TabNet Performance (2016 Held-out Set) ")
print("="*50)
print(f"MAE : {mean_absolute_error(y_test, preds):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.4f}")
print(f"R2  : {r2_score(y_test, preds):.4f}")

# Bootstrap CI Output
bootstrap_results = calculate_bootstrap_ci(y_test, preds, n_bootstraps=1000)
print("\n" + "="*50)
print(" 95% Bootstrap Confidence Intervals (1000 iter) ")
print("="*50)
print(f"MAE  : {bootstrap_results['MAE']}")
print(f"RMSE : {bootstrap_results['RMSE']}")
print(f"R2   : {bootstrap_results['R2']}")

# ------------------------------------------
# Feature Importance Plot
# ------------------------------------------
plt.figure(figsize=(10, 6))

importances = pd.Series(clf_final.feature_importances_, index=feature_names)
top_imp = importances.nlargest(15).sort_values()

# Highlight missingness masks in red
colors = ['#d63031' if '_is_missing' in name else '#0984e3' for name in top_imp.index]

ax = top_imp.plot(kind='barh', color=colors, edgecolor='0.25', linewidth=0.6)
ax.set_xlabel("Importance", fontsize=11)
ax.grid(axis='x', linestyle='--', alpha=0.25, linewidth=0.8)

# Clean aesthetics
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_alpha(0.5)
ax.spines['bottom'].set_alpha(0.5)

# Value annotations
xmax = float(top_imp.max())
for i, v in enumerate(top_imp.values):
    ax.text(v + xmax * 0.01, i, f"{v:.4f}", va='center', fontsize=9)

ax.text(0.99, 0.02, "Red: Missingness Mask\nBlue: Original Feature",
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10, alpha=0.85)

plt.title("MA-TabNet Feature Importance (Top 15)", pad=20, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Advanced Explainable AI (XAI) - TabNet Attention & SHAP

To deeply understand the model's decision-making process, we utilize two distinct explainability approaches:
1. **TabNet Internal Attention:** Visualizes the inherent feature selection masks learned by the TabNet architecture across different decision steps.
2. **SHAP (SHapley Additive exPlanations):** Provides game-theoretic estimations of global feature importance and local sample-level explanations (Beeswarm, Bar, and Waterfall plots).

In [ ]:
# ==============================================================================
# [5] Explainable AI: TabNet Attention & SHAP Analysis
# ==============================================================================
try:
    import shap
    import seaborn as sns
except ImportError:
    %pip install shap seaborn -q
    import shap
    import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ---------------------------------------------------------
# Visualization Settings
# ---------------------------------------------------------
sns.set_theme(style="whitegrid", context="talk", font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'font.family': 'sans-serif',
})

PALETTE_GLOBAL = "viridis"
PALETTE_LOCAL = "mako"

# ---------------------------------------------------------
# 1. TabNet Attention Visualization
# ---------------------------------------------------------
def tabnet_attention_xai_hq(model, X, feature_names, top_k=15, sample_idx=0, save_dir=".", dpi=600):
    """
    Extracts and visualizes high-quality TabNet attention masks.
    """
    os.makedirs(save_dir, exist_ok=True)

    out = model.explain(X)
    if isinstance(out, tuple) and len(out) == 2:
        M_explain, masks = out
    else:
        raise ValueError("Unexpected return format from model.explain(X).")

    step_keys_sorted = sorted(list(masks.keys()))

    # --- Global Attention (Aggregated across steps) ---
    global_sum = np.zeros(len(feature_names))
    for k in step_keys_sorted:
        global_sum += masks[k].mean(axis=0)
    global_sum /= (len(step_keys_sorted) + 1e-12)

    df_global = pd.DataFrame({
        'Feature': feature_names,
        'Importance': global_sum
    }).sort_values(by='Importance', ascending=False).head(top_k)

    fname_global = f"tabnet_global_attention_top{top_k}"
    plt.figure(figsize=(12, 8))
    sns.barplot(data=df_global, x='Importance', y='Feature', palette=PALETTE_GLOBAL, edgecolor=".2")
    plt.xlabel("Mean Attention Score")
    plt.ylabel("")
    plt.title("TabNet Global Attention Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{fname_global}.png"), dpi=dpi, bbox_inches='tight', facecolor="white")
    plt.show()

    # --- Local Attention (For a single sample) ---
    local_sum = np.zeros(len(feature_names))
    for k in step_keys_sorted:
        local_sum += masks[k][sample_idx]
    local_sum /= (len(step_keys_sorted) + 1e-12)

    df_local = pd.DataFrame({
        'Feature': feature_names,
        'Importance': local_sum
    }).sort_values(by='Importance', ascending=False).head(top_k)

    fname_local = f"tabnet_local_attention_sample{sample_idx}_top{top_k}"
    plt.figure(figsize=(12, 8))
    sns.barplot(data=df_local, x='Importance', y='Feature', palette=PALETTE_LOCAL, edgecolor=".2")
    plt.xlabel("Attention Score")
    plt.ylabel("")
    plt.title(f"TabNet Local Attention (Sample Index: {sample_idx})")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{fname_local}.png"), dpi=dpi, bbox_inches='tight', facecolor="white")
    plt.show()

    return df_global, df_local, masks

print(">>> Running TabNet attention analysis...")
df_global_attn, df_local_attn, masks = tabnet_attention_xai_hq(
    clf_final, X_test, feature_names, top_k=15, sample_idx=0
)

# ---------------------------------------------------------
# 2. SHAP Analysis
# ---------------------------------------------------------
print("\n>>> Running SHAP analysis (this may take some time)...")

# Wrapper function for SHAP explainer
def predict_1d(X):
    return clf_final.predict(X).reshape(-1)

rng = np.random.default_rng(0)
# Sample background data to speed up SHAP Permutation Explainer
idx = rng.choice(len(X_train_2), size=min(500, len(X_train_2)), replace=False)
X_bg = X_train_2[idx]

masker = shap.maskers.Independent(X_bg)
explainer = shap.PermutationExplainer(predict_1d, masker, feature_names=feature_names)

# Compute SHAP values for the test set
X_explain = X_test[:150]
shap_values = explainer(X_explain, max_evals=2 * X_explain.shape[1] + 1)

# Plot 1: SHAP Beeswarm Plot (Shows directionality and density)
plt.figure(figsize=(14, 10))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_beeswarm_global_top15.png", dpi=600, bbox_inches='tight', facecolor="white")
plt.show()

# Plot 2: SHAP Bar Plot (Mean Absolute Impact)
plt.figure(figsize=(14, 10))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_bar_global_mean_abs_top15.png", dpi=600, bbox_inches='tight', facecolor="white")
plt.show()

## 6. Permutation Importance

Permutation Feature Importance evaluates the decrease in model performance (MAE degradation) when a single feature's values are randomly shuffled. This breaks the relationship between the feature and the target, highlighting the features the model relies on most heavily.

In [ ]:
# ==============================================================================
# [6] Permutation Importance Visualization
# ==============================================================================
import os
import matplotlib.pyplot as plt
import seaborn as sns

save_dir = "."
dpi = 600
os.makedirs(save_dir, exist_ok=True)

FNAME = "permutation_importance_mae_top15"
OUT_PNG = os.path.join(save_dir, f"{FNAME}.png")
OUT_PDF = os.path.join(save_dir, f"{FNAME}.pdf")

# NOTE: Assumes `df_perm` DataFrame is generated externally or prior to this step.
# It should contain columns: ['Feature', 'Importance', 'Std']
# For demonstration, if df_perm is not defined, we skip plotting.
if 'df_perm' in locals():
    colors = sns.color_palette("crest", n_colors=len(df_perm))

    plt.figure(figsize=(12, 8))

    # Create horizontal bar plot with error bars representing standard deviation
    plt.barh(
        df_perm['Feature'],
        df_perm['Importance'],
        xerr=df_perm['Std'],
        color=colors,
        edgecolor="0.25",
        capsize=4,
        height=0.62,
        error_kw={"elinewidth": 1.2, "alpha": 0.9}
    )

    plt.xlabel("Mean Importance Score (MAE Degradation)", fontsize=11)
    plt.ylabel("")

    # Invert Y-axis to have the most important feature at the top
    plt.gca().invert_yaxis()

    ax = plt.gca()
    ax.grid(True, axis='x', linestyle='--', alpha=0.25, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_alpha(0.6)
    ax.spines["bottom"].set_alpha(0.6)
    ax.tick_params(axis="both", labelsize=10)

    plt.tight_layout()

    plt.savefig(OUT_PNG, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.savefig(OUT_PDF, bbox_inches="tight", facecolor="white")
    plt.show()

    print(f"Saved: {OUT_PNG} (dpi={dpi})")
else:
    print("Variable 'df_perm' is not defined. Please ensure permutation importance calculation is executed before plotting.")

## 7. Partial Dependence Plots (PDP)

Partial Dependence Plots show the marginal effect that one or two features have on the predicted outcome of the model. Here, we visualize the non-linear relationships captured by MA-TabNet for the top influential features.

In [ ]:
# ==============================================================================
# [7] Partial Dependence Plots (PDP)
# ==============================================================================
from sklearn.base import is_classifier, is_regressor
# NOTE: PDP requires a standard sklearn wrapper interface.
# `tabnet_wrapper`, `top_idx`, `pd_one_feature`, and `response_method` must be defined.

# Check estimator type
if 'tabnet_wrapper' in locals():
    print("Estimator Type Check:")
    print("is_classifier:", is_classifier(tabnet_wrapper))
    print("is_regressor :", is_regressor(tabnet_wrapper))

    if hasattr(tabnet_wrapper, "predict_proba"):
        p = tabnet_wrapper.predict_proba(X_test[:10])
        print("predict_proba shape:", np.asarray(p).shape)

    SAVE_PATH_PNG = "PDP_top_features_dpi600.png"
    SAVE_PATH_PDF = "PDP_top_features_dpi600.pdf"

    # Assume top_idx contains the indices of the top features to plot
    features = list(top_idx) if 'top_idx' in locals() else []
    n = len(features)

    if n > 0:
        ncols = min(3, n)
        nrows = int(np.ceil(n / ncols))

        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(5.4 * ncols, 3.6 * nrows),
            squeeze=False
        )
        axes = axes.reshape(-1)

        for i, f in enumerate(features):
            ax = axes[i]

            # Custom function pd_one_feature is assumed to return a dict with 'grid_values' and 'average'
            pd_res = pd_one_feature(
                estimator=tabnet_wrapper,
                X=X_test,
                f=f,
                response_method=response_method if 'response_method' in locals() else 'auto',
                class_idx=class_idx if 'class_idx' in locals() else 0,
                grid_resolution=80
            )

            xs = pd_res["grid_values"][0]
            ys = pd_res["average"][0].reshape(-1)

            # Filter out invalid values
            m = np.isfinite(xs) & np.isfinite(ys)
            xs, ys = xs[m], ys[m]

            ax.plot(xs, ys, linewidth=2.4)
            title = feature_names[f] if feature_names is not None else f"feature {f}"
            ax.set_title(title, fontsize=12, fontweight="semibold", pad=6)
            ax.set_xlabel("Feature value", fontsize=10, labelpad=4)
            ax.set_ylabel("Partial dependence", fontsize=10, labelpad=4)

            # Clean plot aesthetics
            ax.grid(True, linestyle="--", alpha=0.25, linewidth=0.8)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_alpha(0.6)
            ax.spines["bottom"].set_alpha(0.6)
            ax.tick_params(axis="both", labelsize=9)

        # Remove unused subplots
        for j in range(n, len(axes)):
            fig.delaxes(axes[j])

        fig.subplots_adjust(left=0.06, right=0.995, bottom=0.14, top=0.88, wspace=0.18, hspace=0.30)

        fig.savefig(SAVE_PATH_PNG, dpi=600, bbox_inches="tight", facecolor="white")
        fig.savefig(SAVE_PATH_PDF, bbox_inches="tight", facecolor="white")

        plt.show()
        plt.close(fig)
        print(f"Saved: {SAVE_PATH_PNG} (dpi=600)")
else:
    print("Warning: Required wrapper variables (e.g., tabnet_wrapper, top_idx) are not defined. Skipping PDP generation.")